# Bonus: one model, both senses

In Part 3, whisper turned your voice into text and the brain only ever read that text.
Here **one model gets the raw audio and the camera frame in the same request**. No
transcript in between.

| | Part 3 (pipeline) | This notebook (omni) |
|---|---|---|
| Models | whisper + vision LLM | one model |
| What the brain gets | text of what you said | your actual voice + the picture |
| Hears tone, laughter, a door slam | no | yes |
| Transcript you can log | yes, for free | only if you ask for one |

**You need a model that can hear.** Gemma 4 (`"gemma-4-E2B"`, the default, or `"gemma-4-E4B"`)
and Qwen2.5-Omni (`"qwen2.5-omni-3B"`) can. SmolVLM can't: switch in `1_setup.ipynb`, Step 0.

In [ ]:
# Run me first (Shift+Enter). Re-run me if you restart the kernel.
%load_ext autoreload
%autoreload 2

import config
from agent import nb, ears, mouth
from agent.llm import LocalLLM, system, user, assistant

llm = LocalLLM()
print("brain:", llm.model_name() if llm.health() else f"NOT RUNNING at {config.SERVER_URL} (run the Step 0 cells in 1_setup.ipynb)")

In [ ]:
import time
import numpy as np
from agent.eyes import Eyes, preprocess
from agent.llm import audio_part

CAMERA = config.CAMERA
eyes = Eyes(CAMERA)
nb.show(eyes.frame(), width=300)

## Can your brain hear?

We play it a one-second beep. A model that can't take audio will refuse.

In [ ]:
beep = (0.3 * np.sin(np.arange(16000) / 16000 * 2 * np.pi * 880)).astype(np.float32)
nb.play(beep)
try:
    r = llm.chat([user("Describe this sound in a few words.", audio=[ears.to_wav_bytes(beep)])],
                 stream=False, max_tokens=30, temperature=0)
    print("it hears:", r.text.strip(), "|", r.stats())
except RuntimeError as e:
    print("this model can't hear.", str(e)[:300])

## Ask with your voice, about what it sees

Hold something up, run the cell, and ask a question out loud: *"what am I holding?"*,
*"is this ripe?"*, *"read this for me"*. The frame and your voice go in one request.

In [ ]:
INSTRUCTION = ("The audio is the user talking to you. The image is what your webcam sees right now. "
               "Answer what they asked, using the image when it helps. One or two short sentences.")

audio = nb.record(5)                 # or: audio = ears.load_wav("clip.wav")
frame = eyes.frame()
jpeg = preprocess(frame)
nb.show(frame, width=300)

voice = mouth.StreamingMouth()
def on_token(piece):
    nb.stream(piece)
    voice.feed(piece)

r = llm.chat([user(INSTRUCTION, images=[jpeg], audio=[ears.to_wav_bytes(audio)])],
             on_token=on_token, max_tokens=120)
voice.finish()
print("\n\n", r.stats())

## Race: pipeline vs omni

Same recording, same frame, two architectures. Which answers first? Which answer is better?
(Needs whisper from Part 3.)

In [ ]:
wav = ears.to_wav_bytes(audio)

# A) Part 3 style: whisper first, then the brain reads text
t0 = time.perf_counter()
said, _ = ears.transcribe(audio)
ra = llm.chat([user(f"{said}\n(Answer in one or two short sentences.)", images=[jpeg])],
              stream=False, max_tokens=120)
pipeline_s = time.perf_counter() - t0

# B) omni: raw audio straight in
t0 = time.perf_counter()
rb = llm.chat([user(INSTRUCTION, images=[jpeg], audio=[wav])], stream=False, max_tokens=120)
omni_s = time.perf_counter() - t0

print(f"pipeline {pipeline_s:5.2f}s  whisper heard {said!r}\n          -> {ra.text.strip()}\n")
print(f"omni     {omni_s:5.2f}s\n          -> {rb.text.strip()}")

## Things a transcript can't carry

Whisper writes down *words*. An omni model gets the whole sound. Pick an experiment,
record, run. Then write your own.

In [ ]:
EXPERIMENTS = {
    "mood":     "Ignore the words. How does the speaker sound: happy, tired, excited, bored? What gives it away?",
    "sounds":   "Ignore any speech. What other sounds can you hear? Clapping, typing, music, a door?",
    "language": "What language is the speaker using? Translate what they said into English.",
    "liar":     "Does what the speaker says match what you see in the image? If they claim something that isn't true, call it out.",
    "mine":     "Write your own experiment here.",
}
PICK = "mood"     # try each one. "liar": say "I'm holding a banana" while holding a pen

clip = nb.record(5)
r = llm.chat([user(EXPERIMENTS[PICK], images=[preprocess(eyes.frame())], audio=[ears.to_wav_bytes(clip)])],
             on_token=nb.stream, max_tokens=120)
print("\n\n", r.stats())

## A conversation (the model is its own whisper)

Sending every old audio clip again each turn would be slow. So we use the Part 4 trick:
the model starts every reply with a hidden `HEARD:` line. We keep that text as the
memory of what you said, strip it from the voice, and only the new clip is audio.
Run `omni_talk()` a few times.

In [ ]:
import re

OMNI_PROMPT = ("You can see through the user's webcam and hear their voice. "
               "Start every reply with one line: 'HEARD: <exactly what the user just said>'. "
               "Then answer in one to three short sentences, no lists, no markdown.")
HEARD_RE = re.compile(r"(?im)^\s*HEARD:\s*(.*)$")
omni_history = [system(OMNI_PROMPT)]

def omni_talk(seconds=5):
    clip = nb.record(seconds)
    msgs = omni_history + [user("(the user's voice is in the audio)",
                                images=[preprocess(eyes.frame())], audio=[ears.to_wav_bytes(clip)])]
    r = llm.chat(msgs, stream=False, max_tokens=160)
    m = HEARD_RE.search(r.text)
    heard = m.group(1).strip() if m else "(audio)"
    answer = HEARD_RE.sub("", r.text).strip()
    print(f"it heard: {heard}\nit says : {answer}\n{r.stats()}")
    mouth.speak(answer)
    omni_history.append(user(heard))          # cheap text memory instead of old audio
    omni_history.append(assistant(answer))

omni_talk()

In [ ]:
omni_talk()   # again. Ask about something you said last time.

## Try this

1. Run the race three times. Is one architecture always faster on your laptop? Why might that change on a phone?
2. Whisper `tiny.en` vs omni on a noisy recording (music playing). Who wins?
3. "mood" experiment: say the same sentence happy, then bored. Does the answer change?
   A transcript would be identical both times.
4. Swap models in `1_setup.ipynb`: Gemma 4 E2B vs Qwen2.5-Omni 3B. Same questions, different ears.
5. Clips longer than ~30 seconds: what happens? (Some audio encoders only take short windows.)

In [ ]:
eyes.close()